---

# 🔐 Introducción a la criptografía cuántica

### El protocolo BB84, paso a paso

**INTRODUCCIÓN A LA PROGRAMACIÓN CUÁNTICA · CENIDET**  
**TECNOLÓGICO NACIONAL DE MÉXICO**

`NIVEL INICIAL` · `QISKIT` · `GOOGLE COLAB`

---

**Oscar Alejandro López Campero**  
Agosto 2026

---

## 🧭 Ruta del módulo

Avanzaremos lentamente y construiremos una idea a la vez.

| Parte | Pregunta |
|:--:|---|
| **A** | ¿Qué problema resuelve BB84? |
| **B** | ¿Cuáles son las bases Z y X? |
| **C** | ¿Cómo prepara Alice un qubit? |
| **D** | ¿Cómo mide Bob? |
| **E** | ¿Cómo se forma una clave? |
| **F** | ¿Cómo se detecta a Eve? |

> BB84 no envía directamente un mensaje secreto. Primero permite que Alice y Bob generen una clave compartida.

---

# 🛠️ Preparación

Ejecuta estas celdas antes de comenzar.

In [ ]:
# Instalar Qiskit, Aer y las herramientas de visualización.
%pip install -q "qiskit[visualization]" qiskit-aer pandas

In [ ]:
# Herramientas para circuitos, simulación, tablas y gráficas.
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import pandas as pd
import random
import matplotlib.pyplot as plt

simulador = AerSimulator()
print("✅ Entorno preparado")

---

# 🟦 PARTE A

# 1️⃣ ¿Qué quiere lograr BB84?

Alice y Bob quieren obtener la misma clave secreta, por ejemplo:

```text
Alice: 10110
Bob:   10110
```

Una tercera persona, llamada **Eve**, podría intentar escuchar el canal.

La idea especial de BB84 es que medir un qubit con la base incorrecta puede modificarlo. Esa alteración produce errores que Alice y Bob pueden detectar.

## Los tres participantes

| Persona | Función |
|:--:|---|
| 👩 **Alice** | Prepara y envía los qubits |
| 👨 **Bob** | Recibe y mide los qubits |
| 🕵️ **Eve** | Intenta interceptar la comunicación |

```text
Alice  ───── qubits ─────▶  Bob
              ▲
             Eve
```

---

# 🟩 PARTE B

# 2️⃣ Dos formas de representar un bit

BB84 utiliza dos bases:

| Base | Bit 0 | Bit 1 | Forma de medir |
|:--:|:--:|:--:|---|
| **Z** | ∣0⟩ | ∣1⟩ | Medición normal |
| **X** | ∣+⟩ | ∣−⟩ | H y después medición |

La base es como el “idioma” utilizado para preparar o leer el qubit.

## Regla principal

- Si Alice y Bob usan la **misma base**, Bob recupera correctamente el bit.
- Si usan **bases diferentes**, el resultado de Bob es aleatorio.

| Alice | Bob | ¿Qué sucede? |
|:--:|:--:|---|
| Z | Z | Bob recupera el bit |
| X | X | Bob recupera el bit |
| Z | X | Resultado aleatorio |
| X | Z | Resultado aleatorio |

---

# 🟨 PARTE C

# 3️⃣ Los cuatro estados que prepara Alice

Alice elige dos cosas para cada qubit:

1. un bit: `0` o `1`;
2. una base: `Z` o `X`.

Veremos las cuatro combinaciones por separado.

## 3.1 Bit 0 en la base Z → `|0⟩`

No necesitamos aplicar ninguna compuerta porque todos los qubits comienzan en `|0⟩`.

In [ ]:
# Alice quiere enviar el bit 0 usando la base Z.
estado_0_z = QuantumCircuit(1)
vector_0_z = Statevector.from_instruction(estado_0_z)

display(estado_0_z.draw('mpl'))
display(vector_0_z.draw('latex'))
display(plot_bloch_multivector(vector_0_z, title="Bit 0 en base Z"))

## 3.2 Bit 1 en la base Z → `|1⟩`

Aplicamos X para cambiar `|0⟩` por `|1⟩`.

In [ ]:
# Alice quiere enviar el bit 1 usando la base Z.
estado_1_z = QuantumCircuit(1)
estado_1_z.x(0)
vector_1_z = Statevector.from_instruction(estado_1_z)

display(estado_1_z.draw('mpl'))
display(vector_1_z.draw('latex'))
display(plot_bloch_multivector(vector_1_z, title="Bit 1 en base Z"))

## 3.3 Bit 0 en la base X → `|+⟩`

Aplicamos H al estado inicial.

In [ ]:
# Alice quiere enviar el bit 0 usando la base X.
estado_0_x = QuantumCircuit(1)
estado_0_x.h(0)
vector_0_x = Statevector.from_instruction(estado_0_x)

display(estado_0_x.draw('mpl'))
display(vector_0_x.draw('latex'))
display(plot_bloch_multivector(vector_0_x, title="Bit 0 en base X"))

## 3.4 Bit 1 en la base X → `|−⟩`

Primero aplicamos X para preparar `|1⟩` y después H para cambiar a la base X.

In [ ]:
# Alice quiere enviar el bit 1 usando la base X.
estado_1_x = QuantumCircuit(1)
estado_1_x.x(0)
estado_1_x.h(0)
vector_1_x = Statevector.from_instruction(estado_1_x)

display(estado_1_x.draw('mpl'))
display(vector_1_x.draw('latex'))
display(plot_bloch_multivector(vector_1_x, title="Bit 1 en base X"))

## Receta de Alice

| Bit | Base | Compuertas |
|:--:|:--:|---|
| 0 | Z | Ninguna |
| 1 | Z | X |
| 0 | X | H |
| 1 | X | X y después H |

> Alice no anuncia todavía qué base utilizó.

---

# 🟧 PARTE D

# 4️⃣ ¿Cómo mide Bob?

Bob también elige al azar una base para cada qubit.

- Para medir en **Z**, mide directamente.
- Para medir en **X**, aplica H y después mide.

## Ejemplo 1 — Alice y Bob usan la misma base

Alice enviará el bit `1` en la base X. Bob también elegirá X.

In [ ]:
# Crear un qubit y un bit clásico para guardar la medición.
misma_base = QuantumCircuit(1, 1)

# ALICE: preparar el bit 1 en la base X.
misma_base.x(0)
misma_base.h(0)
misma_base.barrier()

# BOB: elegir la base X.
# H permite medir en esa base.
misma_base.h(0)
misma_base.measure(0, 0)

display(misma_base.draw('mpl'))

In [ ]:
# Ejecutar el mismo experimento 100 veces.
misma_base_preparada = transpile(misma_base, simulador)
conteos_misma_base = simulador.run(
    misma_base_preparada,
    shots=100,
    seed_simulator=10
).result().get_counts()

print(conteos_misma_base)
display(plot_histogram(
    conteos_misma_base,
    title="Alice X y Bob X"
))

Bob obtiene siempre `1` porque utilizó la misma base que Alice.

## Ejemplo 2 — Alice y Bob usan bases diferentes

Alice volverá a enviar el bit `1` en X, pero Bob medirá directamente en Z.

In [ ]:
bases_diferentes = QuantumCircuit(1, 1)

# ALICE: preparar el bit 1 en la base X.
bases_diferentes.x(0)
bases_diferentes.h(0)
bases_diferentes.barrier()

# BOB: medir directamente porque eligió Z.
bases_diferentes.measure(0, 0)

display(bases_diferentes.draw('mpl'))

In [ ]:
# Ahora aparecen tanto ceros como unos.
bases_diferentes_preparada = transpile(bases_diferentes, simulador)
conteos_bases_diferentes = simulador.run(
    bases_diferentes_preparada,
    shots=100,
    seed_simulator=10
).result().get_counts()

print(conteos_bases_diferentes)
display(plot_histogram(
    conteos_bases_diferentes,
    title="Alice X y Bob Z"
))

> El qubit no incluye una etiqueta que diga qué base utilizó Alice. Bob debe elegir una base antes de conocerla.

---

# 🟪 PARTE E

# 5️⃣ Construir una clave con varios qubits

El protocolo completo repite el mismo proceso muchas veces:

1. Alice elige bits aleatorios.
2. Alice elige bases aleatorias.
3. Alice prepara y envía los qubits.
4. Bob elige bases y mide.
5. Alice y Bob anuncian únicamente sus bases.
6. Conservan las posiciones donde las bases coincidieron.

Este último proceso se llama **filtrado** o **sifting**.

## Una pequeña función auxiliar

La siguiente función es una receta reutilizable para preparar un bit de Alice y medirlo con la base elegida por Bob.

No necesitamos memorizarla; solamente seguiremos sus comentarios.

In [ ]:
def enviar_un_bit(bit, base_alice, base_bob, semilla):
    """Preparar un qubit con Alice y medirlo con Bob."""

    circuito = QuantumCircuit(1, 1)

    # ALICE: X representa el bit 1.
    if bit == 1:
        circuito.x(0)

    # ALICE: H cambia de la base Z a la base X.
    if base_alice == 'X':
        circuito.h(0)

    circuito.barrier()

    # BOB: para medir en X, primero aplica H.
    if base_bob == 'X':
        circuito.h(0)

    circuito.measure(0, 0)

    # Ejecutar una sola vez porque cada qubit representa un envío.
    preparado = transpile(circuito, simulador)
    conteos = simulador.run(
        preparado,
        shots=1,
        seed_simulator=semilla
    ).result().get_counts()

    # get_counts devuelve algo como {'0': 1}.
    resultado = int(next(iter(conteos)))
    return resultado

## Paso 1 — Alice elige bits y bases

Usaremos ocho posiciones para poder observar todo en una tabla.

In [ ]:
# Bits secretos elegidos por Alice.
bits_alice = [1, 0, 1, 1, 0, 0, 1, 0]

# Base utilizada por Alice para cada bit.
bases_alice = ['X', 'Z', 'Z', 'X', 'X', 'Z', 'X', 'Z']

tabla_alice = pd.DataFrame({
    'Posición': range(1, 9),
    'Bit de Alice': bits_alice,
    'Base de Alice': bases_alice
})

display(tabla_alice)

## Paso 2 — Bob elige sus propias bases

Bob todavía no conoce las bases de Alice.

In [ ]:
# Bases elegidas por Bob de forma independiente.
bases_bob = ['X', 'X', 'Z', 'Z', 'X', 'Z', 'Z', 'Z']

print("Bases de Alice:", bases_alice)
print("Bases de Bob:  ", bases_bob)

## Paso 3 — Enviar y medir cada qubit

Repetimos la receta una vez por cada posición.

In [ ]:
resultados_bob = []

# i toma los valores 0, 1, 2, ..., 7.
for i in range(len(bits_alice)):
    resultado = enviar_un_bit(
        bits_alice[i],
        bases_alice[i],
        bases_bob[i],
        semilla=100 + i
    )
    resultados_bob.append(resultado)

print("Resultados de Bob:", resultados_bob)

## Paso 4 — Comparar solamente las bases

Alice y Bob anuncian públicamente si utilizaron Z o X. **No anuncian sus bits.**

Conservan una posición únicamente cuando sus bases coinciden.

In [ ]:
# True significa que las bases coincidieron.
conservar = []

for i in range(len(bits_alice)):
    coinciden = bases_alice[i] == bases_bob[i]
    conservar.append(coinciden)

tabla_bb84 = pd.DataFrame({
    'Posición': range(1, 9),
    'Bit Alice': bits_alice,
    'Base Alice': bases_alice,
    'Base Bob': bases_bob,
    'Resultado Bob': resultados_bob,
    '¿Conservar?': ['Sí' if x else 'No' for x in conservar]
})

display(tabla_bb84)

## Paso 5 — Formar la clave filtrada

Eliminamos las posiciones con bases diferentes.

In [ ]:
clave_alice = []
clave_bob = []

for i in range(len(bits_alice)):
    if conservar[i]:
        clave_alice.append(bits_alice[i])
        clave_bob.append(resultados_bob[i])

print("Clave de Alice:", clave_alice)
print("Clave de Bob:  ", clave_bob)
print("¿Las claves coinciden?", clave_alice == clave_bob)

En un canal ideal y sin intrusos, las claves filtradas coinciden.

Las posiciones descartadas no son errores: simplemente fueron medidas con bases distintas.

---

# 🟥 PARTE F

# 6️⃣ ¿Qué sucede si Eve intercepta?

Eve no conoce las bases de Alice, así que también debe adivinarlas.

Para cada qubit:

1. Eve elige Z o X.
2. Eve mide el qubit.
3. Eve prepara un qubit nuevo con su resultado.
4. Eve lo envía a Bob.

Si Eve elige la base incorrecta, puede alterar el estado.

## Un ejemplo de la perturbación

Supongamos que:

- Alice envía `|+⟩`, que representa 0 en X.
- Eve mide en Z.
- Eve puede obtener 0 o 1 al azar.
- Eve reenvía `|0⟩` o `|1⟩`.
- Bob mide en X y también obtiene un resultado aleatorio.

Aunque Alice y Bob hayan elegido X, la intervención de Eve puede hacer que sus bits sean diferentes.

## Simulación sencilla con Eve

Utilizaremos 40 qubits. La función siguiente representa la regla que ya observamos:

- bases iguales → se conserva el bit;
- bases diferentes → resultado aleatorio.

In [ ]:
def medir_bit(bit, base_preparacion, base_medicion, generador):
    """Simular una medición ideal usando las reglas de BB84."""

    if base_preparacion == base_medicion:
        return bit

    # Si las bases son diferentes, devolver 0 o 1 al azar.
    return generador.randint(0, 1)

In [ ]:
# La semilla permite repetir exactamente este ejemplo.
generador = random.Random(20)
numero_qubits = 40

# Alice crea sus bits y bases.
bits_a = [generador.randint(0, 1) for _ in range(numero_qubits)]
bases_a = [generador.choice(['Z', 'X']) for _ in range(numero_qubits)]

# Eve y Bob eligen bases sin conocer las de Alice.
bases_eve = [generador.choice(['Z', 'X']) for _ in range(numero_qubits)]
bases_b = [generador.choice(['Z', 'X']) for _ in range(numero_qubits)]

resultados_eve = []
resultados_b = []

for i in range(numero_qubits):
    # Eve mide lo que preparó Alice.
    bit_eve = medir_bit(bits_a[i], bases_a[i], bases_eve[i], generador)
    resultados_eve.append(bit_eve)

    # Eve reenvía su resultado usando su propia base.
    bit_bob = medir_bit(bit_eve, bases_eve[i], bases_b[i], generador)
    resultados_b.append(bit_bob)

## Comparar los resultados

Para detectar a Eve, Alice y Bob revisan únicamente las posiciones donde sus propias bases coincidieron.

In [ ]:
filas_eve = []

for i in range(numero_qubits):
    bases_coinciden = bases_a[i] == bases_b[i]
    hay_error = bases_coinciden and bits_a[i] != resultados_b[i]

    filas_eve.append({
        'Posición': i + 1,
        'Bit Alice': bits_a[i],
        'Base Alice': bases_a[i],
        'Base Eve': bases_eve[i],
        'Base Bob': bases_b[i],
        'Bit Bob': resultados_b[i],
        '¿Conservar?': 'Sí' if bases_coinciden else 'No',
        '¿Error?': 'Sí' if hay_error else 'No'
    })

tabla_eve = pd.DataFrame(filas_eve)

# Mostrar las primeras 20 posiciones para que la tabla no sea demasiado larga.
display(tabla_eve.head(20))

## Calcular la tasa de error: QBER

QBER significa **Quantum Bit Error Rate**. Es la proporción de errores dentro de los bits que Alice y Bob conservaron.

$$
QBER=\frac{\text{bits diferentes}}{\text{bits comparados}}\times100
$$

In [ ]:
# Seleccionar únicamente las filas que Alice y Bob conservarían.
filtrados = tabla_eve[tabla_eve['¿Conservar?'] == 'Sí']

# Contar cuántas de esas filas tienen error.
numero_errores = (filtrados['¿Error?'] == 'Sí').sum()
numero_filtrados = len(filtrados)
qber = numero_errores / numero_filtrados

print("Bits filtrados:", numero_filtrados)
print("Errores encontrados:", numero_errores)
print("QBER:", f"{qber * 100:.1f}%")

## Ver el efecto de Eve

En nuestro canal ideal sin Eve, los bits filtrados no presentaron errores. Comparemos ese resultado con la ronda interceptada.

In [ ]:
# Crear una gráfica sencilla con las tasas de error.
figura, eje = plt.subplots(figsize=(6, 4))
eje.bar(
    ['Sin Eve', 'Con Eve'],
    [0, qber * 100],
    color=['seagreen', 'crimson']
)
eje.set_ylabel('QBER (%)')
eje.set_title('Errores observados en la clave filtrada')
eje.set_ylim(0, 35)

# Escribir el valor encima de cada barra.
eje.text(0, 1, '0%', ha='center')
eje.text(1, qber * 100 + 1, f'{qber * 100:.1f}%', ha='center')

display(figura)
plt.close(figura)

En este ejemplo ideal de interceptar y reenviar, Eve introduce una tasa de error cercana al 25 % en la clave filtrada.

En un sistema real también existe ruido. Alice y Bob deben estimar si el nivel de error es aceptable antes de continuar.

## ¿Qué hacen Alice y Bob al final?

1. Revelan una pequeña parte de los bits filtrados.
2. Calculan el QBER.
3. Si el error es demasiado grande, descartan la ronda.
4. Si es aceptable, realizan corrección de errores y reducción de información de Eve.
5. Obtienen una clave final más corta, pero más segura.

> El canal público debe estar autenticado. De lo contrario, Eve podría intentar hacerse pasar por Alice o Bob.

---

# 🧪 Ejercicios

Las respuestas se encuentran únicamente en el solucionario docente.

## Ejercicio 1 — Preparar un estado

Prepara el bit `1` en la base Z.

In [ ]:
ejercicio_1 = QuantumCircuit(1)

# TODO: agrega la compuerta necesaria.
# ejercicio_1.____(0)

estado_e1 = Statevector.from_instruction(ejercicio_1)
display(ejercicio_1.draw('mpl'))
display(estado_e1.draw('latex'))
display(plot_bloch_multivector(estado_e1))

## Ejercicio 2 — Preparar `|+⟩`

Prepara el bit `0` en la base X.

In [ ]:
ejercicio_2 = QuantumCircuit(1)

# TODO: agrega la compuerta necesaria.
# ejercicio_2.____(0)

estado_e2 = Statevector.from_instruction(ejercicio_2)
display(ejercicio_2.draw('mpl'))
display(estado_e2.draw('latex'))
display(plot_bloch_multivector(estado_e2))

## Ejercicio 3 — Filtrar una clave

Observa la siguiente ronda:

| Posición | 1 | 2 | 3 | 4 | 5 | 6 |
|---|:--:|:--:|:--:|:--:|:--:|:--:|
| Bit de Alice | 1 | 0 | 1 | 1 | 0 | 0 |
| Base de Alice | Z | X | X | Z | Z | X |
| Base de Bob | Z | Z | X | X | Z | X |

- ¿Qué posiciones se conservan?
- ¿Cuál es la clave filtrada de Alice?

## Ejercicio 4 — Analizar a Eve

Responde con tus palabras:

1. ¿Por qué Eve no puede medir siempre en la base correcta?
2. ¿Qué sucede cuando utiliza la base equivocada?
3. ¿Qué valor calculan Alice y Bob para buscar alteraciones?

---

# ✅ Cierre del módulo

En este notebook aprendimos que:

- BB84 distribuye una clave, no el mensaje final;
- Alice codifica cada bit usando Z o X;
- Bob elige una base antes de conocer la de Alice;
- solamente se conservan las posiciones con bases iguales;
- una medición con la base incorrecta puede alterar el qubit;
- Alice y Bob utilizan el QBER para buscar esa alteración.

## 📚 Referencias

- Bennett, C. H. y Brassard, G. (1984). *Quantum cryptography: Public key distribution and coin tossing*.
- [IBM Quantum Learning — Quantum key distribution](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/qkd)

---